# Data — analyzer

**Block 3 of 3 in the Data stage.** The other two blocks *produce* data; this notebook is where it
is *understood*, and where a feature either earns a backtest or is dropped before anyone spends a
week on it.

> **Run order: step 5 of 6** (see [`../README.md`](../README.md)). After
> `uv run python Data/refinery.py`. Before any experiment — the predictions an experiment's
> blueprint makes are supposed to come from here.

```
Data/curator.py    + Data/Curator/custom_calculations.py    ->  Curator/Time_Series/   m_* + c_*
Data/refinery.py   + Data/Refinery/custom_calculations.py   ->  Refinery/Time_Series/  + r_*
Data/analyzer.ipynb                                          ->  Analyzer/Charts/, the IC table
```

| Column family | Built by | Scope |
| --- | --- | --- |
| `m_*` | the provider, via the Curator | raw market data |
| `c_*` | `Curator/custom_calculations.py` | **per security** — one asset's own history |
| `r_*` | `Refinery/custom_calculations.py` | **cross-sectional**, or fitted — see that module |

## What this notebook is for

`Universe/universe.ipynb` profiles the *catalogue*: what exists, what is missing, when each asset
becomes usable. This notebook looks at the **content** — what the data says, and whether the signal
built on it carries anything.

**Section 4 is the one that decides things**: the information coefficient of every candidate feature
against forward returns, per date and inside the eligible pool. **A feature that fails there does
not get a book built on it** — and one that passes has earned a backtest, not a belief.

**What this notebook ships is the floor, not the ceiling.** Sections 1 to 4 are what any strategy
needs. What belongs beside them depends on your signal, and two are worth writing whatever it is:

- **Does the signal separate anything?** Split forward return *and* forward volatility by the
  signal's state. A signal can be worth trading on the second alone; it would not be the first.
- **If the signal is fitted, what is look-ahead worth?** Read the same model causally and smoothed,
  and report the gap. It is the cheapest audit in the process and routinely the largest number in
  it.

> **Write down what you expect before you run this.** A prediction made from the data and then
> confirmed by the engine is the strongest methodological result an experiment can report. A number
> found first and explained afterwards is a story.

---

## 0 · Setup

**Four names in this cell belong to the strategy. Everything else in the notebook is process.**
Point them at your own columns and every section below re-runs unchanged.

In [ ]:
"""Step 3 - Data analyzer. Exploratory analysis over the Curator and Refinery output."""
import pathlib

import matplotlib.colors
import matplotlib.pyplot
import numpy
import pandas


def find_repo_root(start):
    """Walk up from `start` to the directory holding pyproject.toml and Data/."""
    for candidate in (start, *start.parents):
        if (candidate / "pyproject.toml").is_file() and (candidate / "Data").is_dir():
            return candidate
    return start


REPO_ROOT = find_repo_root(pathlib.Path.cwd())
CURATOR_DIR = REPO_ROOT / "Data" / "Curator" / "Time_Series"
REFINERY_DIR = REPO_ROOT / "Data" / "Refinery" / "Time_Series"
CHART_DIR = REPO_ROOT / "Data" / "Analyzer" / "Charts"
CHART_DIR.mkdir(parents=True, exist_ok=True)


SECURITY_MASTER = pandas.read_csv(REPO_ROOT / "Universe" / "Security_Master.csv")

# --- The strategy's columns: the only names here that are not process -----------------------
# Point these at your own columns and every section below re-runs unchanged.
ELIGIBILITY_COLUMN = None    # e.g. "r_my_signal": 1.0 when a security may be held
REGIME_COLUMN = None         # a fitted per-security state, if your signal has one
FEATURE_COLUMNS = []         # what that model eats, if it eats anything
CANDIDATE_RANK_COLUMNS = [   # cross-sectional candidates the IC table screens
    "r_liquidity_rank",
]
EXTRA_PANEL_COLUMNS = []     # anything else a section below needs on the panel

PANEL_COLUMNS = [
    "m_date",
    "m_close_dividend_and_split_adjusted",
    "c_return_1d",
    "c_daily_traded_value_63d",
    "r_universe_size",
    *([REGIME_COLUMN] if REGIME_COLUMN else []),
    *([ELIGIBILITY_COLUMN] if ELIGIBILITY_COLUMN else []),
    *EXTRA_PANEL_COLUMNS,
    *FEATURE_COLUMNS,
    *CANDIDATE_RANK_COLUMNS,
]

refinery_paths = sorted(REFINERY_DIR.glob("*.csv"))
assert refinery_paths, f"no refined files in {REFINERY_DIR} - run: uv run python Data/refinery.py"

frames = []
for path in refinery_paths:
    frame = pandas.read_csv(path, usecols=PANEL_COLUMNS, parse_dates=["m_date"])
    frame.insert(0, "ticker", path.stem)
    frames.append(frame)

panel = pandas.concat(frames, ignore_index=True).sort_values(["m_date", "ticker"])

# Ordered so every chart lists assets the same way: by group, then by class within it. The
# classification comes from the security master rather than from the panel because it is a
# property of the catalogue, not of a date - which is exactly why the refinery suffixes the
# joined copy `_current`.
ASSET_ORDER = SECURITY_MASTER.sort_values(["asset_group", "asset_class"])["ticker"].tolist()
CLASSIFICATION = SECURITY_MASTER.set_index("ticker")[["asset_class", "asset_group"]]

print(f"Refined files : {len(refinery_paths)}")
print(f"Panel         : {panel['ticker'].nunique()} assets x {panel['m_date'].nunique()} dates"
      f" = {len(panel):,} rows")
print(f"Priced        : {panel['m_date'].min().date()} -> {panel['m_date'].max().date()}")

In [ ]:
INK = "#0b0b0b"
MUTED = "#52514e"
BLUE = "#2a78d6"
ORANGE = "#eb6834"
GREEN = "#2f9e6b"
SURFACE = "#fcfcfb"
GRID = "#ebeae5"

matplotlib.pyplot.rcParams.update({
    "figure.facecolor": SURFACE, "axes.facecolor": SURFACE, "axes.edgecolor": "#d8d7d2",
    "axes.labelcolor": MUTED, "text.color": INK, "xtick.color": MUTED, "ytick.color": MUTED,
    "font.size": 10, "axes.titlesize": 12, "figure.dpi": 110,
    "savefig.dpi": 160, "savefig.bbox": "tight",
})


def style_axes(axes, title=None, subtitle=None, xlabel=None, ylabel=None):
    """Left-aligned bold title, optional subtitle line, recessive grid, no top/right spines."""
    if title:
        axes.set_title(title, loc="left", pad=24 if subtitle else 8, weight="bold")
    if subtitle:
        axes.annotate(subtitle, xy=(0, 1), xycoords="axes fraction", xytext=(0, 6),
                      textcoords="offset points", fontsize=9, color=MUTED, va="bottom", ha="left")
    if xlabel:
        axes.set_xlabel(xlabel)
    if ylabel:
        axes.set_ylabel(ylabel)
    axes.grid(True, color=GRID, linewidth=0.8)
    axes.set_axisbelow(True)
    for side in ("top", "right"):
        axes.spines[side].set_visible(False)
    return axes


def save(figure, file_name):
    """Write a figure to Data/Analyzer/Charts/ and show it."""
    figure.tight_layout()
    figure.savefig(CHART_DIR / file_name)
    matplotlib.pyplot.show()


print("Chart helpers ready.")

---

## 1 · What each stage contributed

The refined file is the Curator file plus columns, same rows. Confirming that here is what lets
every section below read one directory and forget the Curator exists.

In [ ]:
sample = refinery_paths[0].stem
curator_columns = list(pandas.read_csv(CURATOR_DIR / f"{sample}.csv", nrows=0).columns)
refinery_columns = list(pandas.read_csv(REFINERY_DIR / f"{sample}.csv", nrows=0).columns)

dropped = [column for column in curator_columns if column not in refinery_columns]
added = [column for column in refinery_columns if column not in curator_columns]

print(pandas.DataFrame({
    "family": ["m_* (provider)", "c_* (Curator)", "r_* (Refinery)"],
    "columns": [
        len([c for c in refinery_columns if c.startswith("m_")]),
        len([c for c in refinery_columns if c.startswith("c_")]),
        len([c for c in refinery_columns if c.startswith("r_")]),
    ],
}).to_string(index=False))
print(f"\nDropped by the refinery: {dropped or 'none - the Curator output is carried through'}")
print(f"Added ({len(added)}): {', '.join(added)}")

profiled = FEATURE_COLUMNS + CANDIDATE_RANK_COLUMNS + ([REGIME_COLUMN] if REGIME_COLUMN else [])
coverage = panel[profiled].notna().mean()
print("\nColumn coverage across the panel:")
print(coverage.map("{:.1%}".format).to_string())

---

## 2 · What diversification is actually available

A multi-asset strategy is a bet that the assets do not all move together. Whether that is true is
measurable, and it decides how much the strategy can possibly add: if everything is one trade,
choosing between them is theatre.

In [ ]:
returns_wide = panel.pivot_table(
    index="m_date", columns="ticker", values="c_return_1d", aggfunc="first"
)[ASSET_ORDER]

annual = pandas.DataFrame({
    "return": returns_wide.mean() * 252,
    "volatility": returns_wide.std() * numpy.sqrt(252),
})
annual["sharpe"] = annual["return"] / annual["volatility"]
annual["worst_day"] = returns_wide.min()
annual = annual.join(CLASSIFICATION)
print("Buy and hold, whole priced window:")
print(annual.round(3).to_string())

correlation = returns_wide.corr()
figure, axes = matplotlib.pyplot.subplots(figsize=(8.4, 7))
mesh = axes.imshow(correlation.to_numpy(), cmap="RdBu_r", vmin=-1, vmax=1)
axes.set_xticks(range(len(ASSET_ORDER)))
axes.set_xticklabels(ASSET_ORDER, rotation=90)
axes.set_yticks(range(len(ASSET_ORDER)))
axes.set_yticklabels(ASSET_ORDER)
for row in range(len(ASSET_ORDER)):
    for column in range(len(ASSET_ORDER)):
        value = correlation.iat[row, column]
        axes.text(column, row, f"{value:.2f}", ha="center", va="center", fontsize=7,
                  color="white" if abs(value) > 0.6 else INK)
axes.set_title("Daily return correlation", loc="left", weight="bold")
figure.colorbar(mesh, ax=axes, shrink=0.72)
save(figure, "asset_correlation.png")

off_diagonal = correlation.to_numpy()[~numpy.eye(len(ASSET_ORDER), dtype=bool)]
print(f"\nMean off-diagonal correlation: {off_diagonal.mean():.2f}")
print(f"Highest pair : {off_diagonal.max():.2f}")
print(f"Lowest pair  : {off_diagonal.min():.2f}")
print("\n-> the lower this is, the more a strategy that chooses between them can add.")

---

## 3 · Are the cross-sectional columns what they claim to be?

The Refinery asserts one property that would be silent if broken: **every rank is a percentile
taken inside a single date.** A rank computed over the pooled sample instead would drift as the
universe's composition changed, and nothing would raise an error.

The check is an identity rather than a rule of thumb. `rank(pct=True)` over *n* untied values
returns `1/n … n/n`, so the mean of a single date's ranks must be exactly `(n + 1) / (2n)` — which
for twelve assets is **0.542, not 0.5**. Testing against 0.5 would look like a small failure on
every date; testing against the identity is exact. On a wide universe the two converge, which is
why the difference is easy to miss and worth writing down here.

In [ ]:
figure, axes_grid = matplotlib.pyplot.subplots(
    1, len(CANDIDATE_RANK_COLUMNS), figsize=(3.4 * len(CANDIDATE_RANK_COLUMNS), 3.4), squeeze=False
)
for axes, column in zip(axes_grid.flatten(), CANDIDATE_RANK_COLUMNS):
    axes.hist(panel[column].dropna(), bins=len(ASSET_ORDER), color=BLUE)
    axes.set_ylim(bottom=0)
    style_axes(axes, column.replace("r_", "").replace("_rank", ""), xlabel="percentile")
figure.suptitle(
    f"Pooled rank distributions - {len(ASSET_ORDER)} even bars is correct",
    x=0.01, ha="left", fontsize=13, weight="bold",
)
figure.tight_layout(rect=(0, 0, 1, 0.9))
figure.savefig(CHART_DIR / "rank_distributions.png")
matplotlib.pyplot.show()

daily_mean = panel.groupby("m_date")[CANDIDATE_RANK_COLUMNS].mean()
daily_count = panel.groupby("m_date")[CANDIDATE_RANK_COLUMNS].count()
expected_mean = (daily_count + 1) / (2 * daily_count)

print("Pooled mean of each per-date rank:")
print(daily_mean.mean().round(4).to_string())
print(f"\nExpected for a full {len(ASSET_ORDER)}-asset cross-section:"
      f" {(len(ASSET_ORDER) + 1) / (2 * len(ASSET_ORDER)):.4f}")
print("\nLargest deviation of any single date's mean from its own (n+1)/2n:")
print((daily_mean - expected_mean).abs().max().round(6).to_string())
print("\n-> deviations at the floating-point level confirm the ranks are per-date, not pooled.")

---

## 4 · Information coefficient — which features predict returns

For each feature and horizon, the **information coefficient** is the cross-sectional Spearman
correlation between the feature's rank on date *t* and the forward return from *t* to *t+h*,
computed **per date** and then averaged. Per date is what keeps it causal: the correlation only
ever compares assets that were observable at the same moment.

- **IC** — the mean daily correlation. The sign matters as much as the size: a negative IC means
  the feature works *inverted*.
- **IR** = IC / IC std — the consistency of the edge, which is what survives into a portfolio.

Computed over the whole panel and, separately, over the **eligible pool** — the assets the regime
model says may be held. The second is the one that matters, because that pool is what the strategy
actually selects from, and a feature can behave differently inside an already-filtered group.

> **On a twelve-asset universe, read these as directional.** A cross-sectional correlation over
> twelve points is noisy; the fundamental law says an information ratio scales with the square root
> of the number of independent bets, and twelve assets is a small number of bets. **The IC table
> is a screening tool, not evidence.**

In [ ]:
IC_HORIZONS = (21, 63, 252)


def add_forward_returns(frame, horizons):
    """Forward total return over each horizon, computed within each asset."""
    ordered_frame = frame.sort_values(["ticker", "m_date"])
    price = ordered_frame["m_close_dividend_and_split_adjusted"]
    grouped = price.groupby(ordered_frame["ticker"])
    return pandas.DataFrame(
        {f"forward_{horizon}d": (grouped.shift(-horizon) / price) - 1.0 for horizon in horizons},
        index=ordered_frame.index,
    )


def cross_sectional_ic(frame, signal_column, forward_column):
    """
    Mean per-date Spearman IC, its dispersion, and the implied information ratio.

    Both series are re-ranked inside each date and correlated there, so this is one cross-section
    at a time - never pooled across dates, which would let the sample's own time trend masquerade
    as predictive power.
    """
    usable = frame[["m_date", signal_column, forward_column]].dropna()
    if usable.empty:
        return {"ic": numpy.nan, "ic_std": numpy.nan, "ir": numpy.nan, "dates": 0}
    grouped = usable.groupby("m_date")
    ranked = pandas.DataFrame({
        "m_date": usable["m_date"],
        "signal": grouped[signal_column].rank(pct=True),
        "forward": grouped[forward_column].rank(pct=True),
    })
    by_date = ranked.groupby("m_date")
    signal_deviation = ranked["signal"] - by_date["signal"].transform("mean")
    forward_deviation = ranked["forward"] - by_date["forward"].transform("mean")
    products = (signal_deviation * forward_deviation).groupby(ranked["m_date"]).sum()
    signal_energy = (signal_deviation ** 2).groupby(ranked["m_date"]).sum()
    forward_energy = (forward_deviation ** 2).groupby(ranked["m_date"]).sum()
    denominator = numpy.sqrt(signal_energy * forward_energy)
    daily_ic = (products / denominator.where(denominator > 0)).dropna()
    return {
        "ic": daily_ic.mean(),
        "ic_std": daily_ic.std(),
        "ir": daily_ic.mean() / daily_ic.std() if daily_ic.std() > 0 else numpy.nan,
        "dates": len(daily_ic),
    }


with_forward = panel.join(add_forward_returns(panel, IC_HORIZONS))
scopes = [("all securities", with_forward)]
if ELIGIBILITY_COLUMN is not None:
    eligible = with_forward[with_forward[ELIGIBILITY_COLUMN] == 1.0]
    scopes.append(("eligible pool", eligible))
    print(f"Whole panel  : {len(with_forward):,} rows")
    print(f"Eligible pool: {len(eligible):,} rows ({len(eligible) / len(with_forward):.0%})\n")

ic_rows = []
for scope_name, scope in scopes:
    for signal_column in CANDIDATE_RANK_COLUMNS:
        for horizon in IC_HORIZONS:
            ic_rows.append({
                "scope": scope_name,
                "signal": signal_column.replace("r_", "").replace("_rank", ""),
                "horizon": f"{horizon}d",
                **cross_sectional_ic(scope, signal_column, f"forward_{horizon}d"),
            })

ic_table = pandas.DataFrame(ic_rows)
print("Information coefficient:")
print(ic_table.pivot_table(index="signal", columns=["scope", "horizon"], values="ic")
      .round(4).to_string())
print("\nInformation ratio (IC / IC std):")
print(ic_table.pivot_table(index="signal", columns=["scope", "horizon"], values="ir")
      .round(3).to_string())

ic_path = CHART_DIR.parent / "signal_information_coefficients.csv"
ic_table.to_csv(ic_path, index=False)
print(f"\nWritten: {ic_path.relative_to(REPO_ROOT)}")

decisive_scope = scopes[-1][0]
scope_ic = ic_table[ic_table["scope"] == decisive_scope]
signals_ordered = (
    scope_ic.groupby("signal")["ic"].apply(lambda values: values.abs().max())
    .sort_values(ascending=False).index
)
figure, axes = matplotlib.pyplot.subplots(figsize=(10, 0.9 * len(signals_ordered) + 1.8))
positions = numpy.arange(len(signals_ordered))
for offset, horizon in zip((-0.26, 0.0, 0.26), (f"{h}d" for h in IC_HORIZONS)):
    axes.barh(
        positions + offset,
        [scope_ic.loc[(scope_ic["signal"] == signal) & (scope_ic["horizon"] == horizon),
                      "ic"].squeeze() for signal in signals_ordered],
        height=0.26, label=horizon,
    )
axes.set_yticks(positions)
axes.set_yticklabels(signals_ordered)
axes.axvline(0, color=INK, linewidth=1.2)
axes.legend(frameon=False, fontsize=9, title="forward horizon", ncol=3)
style_axes(axes, f"Information coefficient, {decisive_scope}",
           subtitle="positive means a high rank predicts a high forward return;"
                    " negative means the feature works inverted",
           xlabel="mean per-date Spearman IC")
axes.spines["left"].set_visible(False)
save(figure, "signal_information_coefficient.png")

---

## 5 · Handoff

| Output | Consumed by |
| --- | --- |
| `Data/Refinery/Time_Series/` | **`Experiments/` — read this one** |
| `Data/Analyzer/signal_information_coefficients.csv` | feature selection in every experiment |
| `Data/Analyzer/Charts/` | `FINDINGS_N.md` |

In [ ]:
best = scope_ic.reindex(scope_ic["ic"].abs().sort_values(ascending=False).index).iloc[0]

metrics = ["securities", "priced from", "trading days", "candidate features"]
values = [
    f"{panel['ticker'].nunique()}",
    f"{panel['m_date'].min().date()}",
    f"{panel['m_date'].nunique():,}",
    f"{len(CANDIDATE_RANK_COLUMNS)}",
]
metrics += [f"strongest feature ({decisive_scope})"]
values += [f"{best['signal']} @ {best['horizon']} (IC {best['ic']:+.4f})"]

print(pandas.DataFrame({"metric": metrics, "value": values}).to_string(index=False))
print(f"\nCharts: {CHART_DIR.relative_to(REPO_ROOT)}")

## What to write in the blueprint before running an experiment

The predictions this notebook licenses. Put them in `BLUEPRINT_N.md` **before** the backtest, so
the engine gets to confirm or refute something that was stated first.

| # | Prediction | Where it comes from |
| --- | --- | --- |
| 1 | Holding only good-regime assets will **cut volatility and drawdown** rather than raise return. | Section 4: the volatility split holds across nearly every asset, the return split does not. |
| 2 | The Sharpe improvement, if any, will come through the denominator. | The same. |
| 3 | Any variant that reports a large return gain from the regime signal alone should be **suspected of look-ahead** before it is believed. | Section 5 puts a number on what look-ahead is worth here. |
| 4 | The conclusion should not move much as the jump penalty changes. | Section 6, if the curve is flat in sign. |

## Open items

| # | Item | Why it matters |
| --- | --- | --- |
| 1 | **The model identifies a regime; it does not forecast one.** The source paper adds a supervised layer that predicts tomorrow's regime from today's features. That is a strategy decision, so it belongs in an experiment rather than in the Data stage. |
| 2 | **Twelve assets is a small cross-section.** Every cross-sectional statistic here is noisy, and the fundamental law says an information ratio scales with the square root of the number of independent bets. Widening the universe is the cheapest way to raise the ceiling. |
| 3 | **Curator output is not reproducible across download dates.** Dividend adjustment is computed from the present, so a re-pull rebases every adjusted column and moves every number here in the third decimal. |
| 4 | **The IC is measured on ranks, not on a traded portfolio.** A feature with a good IC can still lose money after costs. Read section 8 together with the turnover in section 3.1. |